In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ETL Retail") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Modo:", spark.sparkContext.master)

In [ ]:
df = spark.read.csv(
    "hdfs://namenode:9000/datos/retail/online_retail.csv",
    header=True,
    inferSchema=True
)

print(f"Filas: {df.count():,}")
print(f"Columnas: {len(df.columns)}")
df.printSchema()

In [1]:
from pyspark.sql.functions import col, count, when, isnan, isnull

print("=== MUESTRA DE DATOS ===")
df.show(5, truncate=False)

print("\n=== NULOS POR COLUMNA ===")
df.select([
    count(when(isnull(c) | isnan(c), c)).alias(c)
    for c in df.columns
]).show()

print("\n=== ESTADÍSTICAS DESCRIPTIVAS ===")
df.describe(["Quantity", "Price", "Customer ID"]).show()

print("\n=== PAÍSES ÚNICOS ===")
df.select("Country").distinct().count()

ModuleNotFoundError: No module named 'pyspark'

In [ ]:
from pyspark.sql.functions import (
    to_timestamp, year, month, dayofweek, hour, weekofyear,
    col, when, round as spark_round
)

# 1. Parsear InvoiceDate — formato real del CSV
df2 = df.withColumn(
    "InvoiceDateTS",
    to_timestamp(col("InvoiceDate"), "yyyy-MM-dd HH:mm:ss")
)

# 2. Filtrar filas inválidas
df2 = df2.filter(
    (col("Quantity") > 0) &
    (col("Price") > 0) &
    col("InvoiceDateTS").isNotNull()
)

# 3. Variables temporales (numéricas)
df2 = df2 \
    .withColumn("Year",        year(col("InvoiceDateTS"))) \
    .withColumn("Month",       month(col("InvoiceDateTS"))) \
    .withColumn("DayOfWeek",   dayofweek(col("InvoiceDateTS"))) \
    .withColumn("Hour",        hour(col("InvoiceDateTS"))) \
    .withColumn("WeekOfYear",  weekofyear(col("InvoiceDateTS")))

# 4. Variable numérica derivada
df2 = df2.withColumn("TotalRevenue", spark_round(col("Quantity") * col("Price"), 2))

# 5. Variables categóricas derivadas
df2 = df2.withColumn(
    "IsCancelled",
    when(col("Invoice").startswith("C"), "Yes").otherwise("No")
)

df2 = df2.withColumn(
    "PriceRange",
    when(col("Price") < 2,    "Low")
    .when(col("Price") < 10,  "Medium")
    .when(col("Price") < 50,  "High")
    .otherwise("Premium")
)

print(f"Filas después de limpieza: {df2.count():,}")
df2.show(3)

In [ ]:
from pyspark.sql.functions import (
    sum as spark_sum, count, countDistinct, avg, round as spark_round
)

# 1. Ventas por país
ventas_pais = df2.groupBy("Country").agg(
    spark_round(spark_sum("TotalRevenue"), 2).alias("total_revenue"),
    countDistinct("Invoice").alias("num_invoices"),
    spark_round(avg("TotalRevenue"), 2).alias("avg_revenue_per_item"),
    spark_sum("Quantity").alias("total_items_sold")
).orderBy("total_revenue", ascending=False)

# 2. Ventas por mes (serie temporal)
ventas_mes = df2.groupBy("Year", "Month").agg(
    spark_round(spark_sum("TotalRevenue"), 2).alias("total_revenue"),
    countDistinct("Invoice").alias("num_invoices"),
    spark_sum("Quantity").alias("total_items")
).orderBy("Year", "Month")

# 3. Ventas por rango de precio
ventas_precio = df2.groupBy("PriceRange").agg(
    count("*").alias("num_transactions"),
    spark_round(spark_sum("TotalRevenue"), 2).alias("total_revenue"),
    spark_round(avg("Price"), 2).alias("avg_price")
).orderBy("total_revenue", ascending=False)

# 4. Ventas por día de la semana
ventas_dia = df2.groupBy("DayOfWeek").agg(
    spark_round(spark_sum("TotalRevenue"), 2).alias("total_revenue"),
    countDistinct("Invoice").alias("num_invoices")
).orderBy("DayOfWeek")

# 5. Top 20 productos por revenue
top_productos = df2.groupBy("StockCode", "Description").agg(
    spark_round(spark_sum("TotalRevenue"), 2).alias("total_revenue"),
    spark_sum("Quantity").alias("total_sold")
).orderBy("total_revenue", ascending=False).limit(20)

print("Países con más ventas:")
ventas_pais.show(5)
print("Ventas por mes:")
ventas_mes.show(5)

In [ ]:
JDBC_URL = "jdbc:postgresql://postgres:5432/retail_db"
JDBC_PROPS = {
    "user": "retail_user",
    "password": "retail_pass",
    "driver": "org.postgresql.Driver"
}

# Convierte todos los nombres de columna a minúsculas
def lower_cols(df):
    return df.toDF(*[c.lower() for c in df.columns])

tablas = {
    "ventas_por_pais":       lower_cols(ventas_pais),
    "ventas_por_mes":        lower_cols(ventas_mes),
    "ventas_por_precio":     lower_cols(ventas_precio),
    "ventas_por_dia_semana": lower_cols(ventas_dia),
    "top_productos":         lower_cols(top_productos),
}

for nombre, df_tabla in tablas.items():
    df_tabla.write.jdbc(
        url=JDBC_URL,
        table=nombre,
        mode="overwrite",
        properties=JDBC_PROPS
    )
    print(f"✓ Tabla '{nombre}' escrita en PostgreSQL")

print("\n¡ETL completado!")